---

## 5. Modelling

### 5.1 Membangun Algoritma Support Vehicle Machine (SVM)

In [ ]:
import numpy as np

class SVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        n_samples, n_features = X.shape

        y_ = np.where(y <= 0, -1, 1)

        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1

                if condition:
                    self.w -= self.learning_rate * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.learning_rate * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.learning_rate * y_[idx]

    def decision(self, X):
        X = np.asarray(X, dtype=float)
        return np.dot(X, self.w) - self.b

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        approx = np.dot(X, self.w) - self.b
        return np.where(approx >= 0, 1, -1)

### 5.2 Pelatihan Model & Informasi Hyperplane

In [ ]:
model_svm = SVM(learning_rate=0.001, lambda_param=0.01, n_iters=1000)
model_svm.fit(X_train_scaled, y_train)

norma_w = np.linalg.norm(model_svm.w)
margin_width = 2.0 / norma_w

print(f"Bobot Fitur 1 (Tingkat Theta)       : {model_svm.w[0]:.4f}")
print(f"Bobot Fitur 2 (Indeks Kedipan Mata) : {model_svm.w[1]:.4f}")
print(f"Nilai Bias                          : {model_svm.b:.4f}")
print(f"Norma Bobot                         : {norma_w:.4f}")
print(f"Lebar Margin Optimal                : {margin_width:.4f}")
print(f"\nPersamaan Hyperplane:")
print(f"{model_svm.w[0]:.4f} * X1 + ({model_svm.w[1]:.4f}) * X2 - ({model_svm.b:.4f}) = 0")

### 5.3 Memprediksi Data Latih & Data Uji

In [ ]:
import pandas as pd

y_pred_train = model_svm.predict(X_train_scaled)
y_pred_test = model_svm.predict(X_test_scaled)

# y_pred_train_01 = np.where(y_pred_train == -1, 0, 1)
# y_pred_test_01 = np.where(y_pred_test == -1, 0, 1)

df_hasil = pd.DataFrame({
    "Label Aktual": np.where(np.asarray(y_test) <= 0, -1, 1),
    "Label Prediksi": y_pred_test,
    "Status": np.where(y_pred_test == 1, "Mengantuk", "Tidak Mengantuk")
})

print("10 Hasil Prediksi pada Data Testing:")
print(df_hasil.head(10))

---

## 6. Evaluation

### 6.1 Confusion Matrix & Metrik Evaluasi

In [ ]:
def evaluation(y_true, y_pred):
    y_true_biner = np.where(np.asarray(y_true) <= 0, -1, 1)
    y_pred_biner = np.where(np.asarray(y_pred) <= 0, -1, 1)

    TP = int(np.sum((y_true_biner == 1) & (y_pred_biner == 1)))
    TN = int(np.sum((y_true_biner == -1) & (y_pred_biner == -1)))
    FP = int(np.sum((y_true_biner == -1) & (y_pred_biner == 1)))
    FN = int(np.sum((y_true_biner == 1) & (y_pred_biner == -1)))

    total = TP + TN + FP + FN
    accuracy = (TP + TN) / total if total > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    confusion_matrix = np.array([[TN, FP], [FN, TP]])

    return {"TP": TP, "TN": TN, "FP": FP, "FN": FN, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1_score, "Confusion Matrix": confusion_matrix}

eval_train = evaluation(y_train, y_pred_train)
eval_test = evaluation(y_test, y_pred_test)

df_evaluation = pd.DataFrame({
    "Metrik Evaluasi": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Data Training": [f"{eval_train['Accuracy']*100:.2f}%", f"{eval_train['Precision']*100:.2f}%", f"{eval_train['Recall']*100:.2f}%", f"{eval_train['F1-Score']*100:.2f}%"],
    "Data Testing": [f"{eval_test['Accuracy']*100:.2f}%", f"{eval_test['Precision']*100:.2f}%", f"{eval_test['Recall']*100:.2f}%", f"{eval_test['F1-Score']*100:.2f}%"]
})

print(df_evaluation)
print(f"\nConfusion Matrix Testing -> TN: {eval_test['TN']}, FP: {eval_test['FP']}, FN: {eval_test['FN']}, TP: {eval_test['TP']}")

### 6.2 Visualisasi Confusion Matrix dan Perbandingan Metrik

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

confusion_matrix_test = eval_test["Confusion Matrix"]
kelas_label = ["Tidak Mengantuk (-1)", "Mengantuk (1)"]

sns.heatmap(confusion_matrix_test, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=kelas_label, yticklabels=kelas_label, annot_kws={"size": 14, "weight": "bold"}, ax=axes[0], linewidths=1.5, linecolor="white")
axes[0].set_title("Confusion Matrix - Data Testing", fontsize=13, fontweight="bold", pad=12)
axes[0].set_xlabel("Label Prediksi ", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Label Aktual ", fontsize=11, fontweight="bold")

nama_metrik = ["Accuracy", "Precision", "Recall", "F1-Score"]
skor_train = [eval_train[m] * 100 for m in nama_metrik]
skor_test = [eval_test[m] * 100 for m in nama_metrik]

x_posisi = np.arange(len(nama_metrik))
lebar = 0.35

bar_train = axes[1].bar(x_posisi - lebar/2, skor_train, lebar, label="Data Training", color="#2b5c8f")
bar_test = axes[1].bar(x_posisi + lebar/2, skor_test, lebar, label="Data Testing", color="#36a2eb")

axes[1].set_ylim(90, 103)
axes[1].set_ylabel("Skor", fontsize=11, fontweight="bold")
axes[1].set_title("Perbandingan Metrik Evaluasi ", fontsize=13, fontweight="bold", pad=12)
axes[1].set_xticks(x_posisi)
axes[1].set_xticklabels(nama_metrik, fontsize=11, fontweight="bold")
axes[1].legend(loc="lower right")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

for bar in list(bar_train) + list(bar_test):
    tinggi = bar.get_height()
    axes[1].annotate(f"{tinggi:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, tinggi), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9.5, fontweight="bold")

plt.tight_layout()
plt.show()

### 6.3 Visualisasi Batas Keputusan Hyperplane dan Margin

In [ ]:
def hyperplane_value(x, w, b, offset):
    return (-w[0] * x + b + offset) / w[1]

fig, ax = plt.subplots(figsize=(8.5, 6))

X_visualisasi = np.asarray(X_train_scaled, dtype=float)
y_visualisasi = np.where(np.asarray(y_train) <= 0, -1, 1)

tidak_kantuk = (y_visualisasi == -1)
mask_kantuk = (y_visualisasi == 1)

ax.scatter(X_visualisasi[tidak_kantuk, 0], X_visualisasi[tidak_kantuk, 1], c="#2563eb", edgecolors="k", s=45, alpha=0.8, label="Tidak Mengantuk (-1)")

ax.scatter(X_visualisasi[mask_kantuk, 0], X_visualisasi[mask_kantuk, 1], c="#dc2626", edgecolors="k", s=45, alpha=0.8, label="Mengantuk (+1)")

x0_min, x0_max = X_visualisasi[:, 0].min() - 0.3, X_visualisasi[:, 0].max() + 0.3
x0_values = np.linspace(x0_min, x0_max, 200)

x1_decision = hyperplane_value(x0_values, model_svm.w, model_svm.b, 0)
x1_margin_positif = hyperplane_value(x0_values, model_svm.w, model_svm.b, 1)
x1_margin_negatif = hyperplane_value(x0_values, model_svm.w, model_svm.b, -1)

ax.plot(x0_values, x1_decision, "k-", lw=2, label="Decision Boundary ")
ax.plot(x0_values, x1_margin_positif, "k--", lw=1.4, label="Margin Boundary ")
ax.plot(x0_values, x1_margin_negatif, "k--", lw=1.4)

x1_min, x1_max = X_visualisasi[:, 1].min() - 0.5, X_visualisasi[:, 1].max() + 0.5
ax.set_xlim(x0_min, x0_max)
ax.set_ylim(x1_min, x1_max)

ax.set_title(f"SVM Scratch: w=[{model_svm.w[0]:.2f}, {model_svm.w[1]:.2f}], b={model_svm.b:.4f}", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Fitur 1 (Scaled)", fontsize=11, fontweight="bold")
ax.set_ylabel("Fitur 2 (Scaled)", fontsize=11, fontweight="bold")
ax.legend(loc="upper right", frameon=True)
ax.grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

---

## 7. GUI (Graphical User Interface)

In [ ]:
import tkinter as tk
import numpy as np
import matplotlib.pyplot as plt
from tkinter import ttk, messagebox

def jalankan_gui_svm():
    root = tk.Tk()
    root.title("Sistem Deteksi Status Kantuk Pengemudi")
    root.geometry("1020x620")
    root.configure(bg="#f8fafc")
    root.resizable(False, False)

    header_frame = tk.Frame(root, bg="#1e293b", height=65)
    header_frame.pack(fill="x")
    tk.Label(header_frame, text="SISTEM PEMANTAUAN KESELAMATAN & DETEKSI KANTUK PENGEMUDI", font=("Segoe UI", 14, "bold"), fg="white", bg="#1e293b", pady=8).pack()
    tk.Label(header_frame, text="Klasifikasi Berdasarkan Tingkat Otak Theta & Indeks Kedipan Mata", font=("Segoe UI", 9), fg="#cbd5e1", bg="#1e293b").pack(pady=(0, 8))

    main_frame = tk.Frame(root, bg="#f8fafc", padx=16, pady=12)
    main_frame.pack(fill="both", expand=True)

    left_panel = tk.Frame(main_frame, bg="#ffffff", bd=1, relief="solid", padx=16, pady=14, width=410)
    left_panel.pack(side="left", fill="y", padx=(0, 12))
    left_panel.pack_propagate(False)

    right_panel = tk.Frame(main_frame, bg="#ffffff", bd=1, relief="solid", padx=10, pady=10)
    right_panel.pack(side="right", fill="both", expand=True)

    tk.Label(left_panel, text="Input Parameter Sensor Fisiologis", font=("Segoe UI", 11, "bold"), bg="#ffffff", fg="#0f172a").pack(anchor="w", pady=(0, 8))

    tk.Label(left_panel, text="Tingkat Otak Theta :", font=("Segoe UI", 9, "bold"), bg="#ffffff", fg="#334155").pack(anchor="w")
    entry_theta = ttk.Entry(left_panel, font=("Segoe UI", 11))
    entry_theta.pack(fill="x", pady=(3, 10))
    entry_theta.insert(0, "2.50")

    tk.Label(left_panel, text="Indeks Kedipan Mata:",font=("Segoe UI", 9, "bold"), bg="#ffffff", fg="#334155").pack(anchor="w")
    entry_kedipan = ttk.Entry(left_panel, font=("Segoe UI", 11))
    entry_kedipan.pack(fill="x", pady=(3, 12))
    entry_kedipan.insert(0, "24.50")

    preset_frame = tk.Frame(left_panel, bg="#ffffff")
    preset_frame.pack(fill="x", pady=(0, 10))

    def isi_preset(theta_val, kedipan_val):
        entry_theta.delete(0, tk.END)
        entry_theta.insert(0, str(theta_val))
        entry_kedipan.delete(0, tk.END)
        entry_kedipan.insert(0, str(kedipan_val))
        proses_prediksi()

    tk.Button(preset_frame, text="Contoh Waspada", font=("Segoe UI", 8, "bold"), bg="#dbeafe", fg="#1e40af", relief="flat", padx=8, pady=4, command=lambda: isi_preset(2.50, 24.50)).pack(side="left", expand=True, fill="x", padx=(0, 4))

    tk.Button(preset_frame, text="Contoh Mengantuk", font=("Segoe UI", 8, "bold"), bg="#fee2e2", fg="#991b1b", relief="flat", padx=8, pady=4, command=lambda: isi_preset(10.80, 15.80)).pack(side="left", expand=True, fill="x", padx=4)

    tk.Label(left_panel, text="2. Hasil Keputusan", font=("Segoe UI", 11, "bold"), bg="#ffffff", fg="#0f172a").pack(anchor="w", pady=(8, 6))

    hasil_box = tk.Frame(left_panel, bg="#f1f5f9", bd=1, relief="solid", padx=12, pady=12)
    hasil_box.pack(fill="x", pady=(0, 10))

    lbl_status = tk.Label(hasil_box, text="Belum Dianalisis", font=("Segoe UI", 13, "bold"), bg="#f1f5f9", fg="#475569")
    lbl_status.pack()

    lbl_rekomendasi = tk.Label(hasil_box, text="Masukkan nilai sensor lalu klik Deteksi Status Kantuk", font=("Segoe UI", 9), bg="#f1f5f9", fg="#64748b", wraplength=340, justify="center")
    lbl_rekomendasi.pack(pady=(4, 0))

    lbl_detail = tk.Label(left_panel, text="Detail Perhitungan Hyperplane:\n• Fitur Scaled : -\n• Nilai f(x) = w·x - b  : -\n• Jarak ke Hyperplane   : -", font=("Consolas", 9), bg="#f8fafc", fg="#1e293b", justify="left", anchor="w", padx=8, pady=8, relief="groove")
    lbl_detail.pack(fill="x", pady=(0, 12))

    fig, ax = plt.subplots(figsize=(5.3, 4.5), dpi=100)
    canvas = FigureCanvasTkAgg(fig, master=right_panel)
    canvas.get_tk_widget().pack(fill="both", expand=True)

    def gambar_plot(titik_scaled=None, kelas_pred=None):
        ax.clear()
        X_vis = np.asarray(X_train_scaled, dtype=float)
        y_vis = np.where(np.asarray(y_train) <= 0, -1, 1)

        ax.scatter(X_vis[y_vis == -1, 0], X_vis[y_vis == -1, 1], c="#3b82f6", alpha=0.35, s=25, label="Data Latih: Tidak Mengantuk") 
        ax.scatter(X_vis[y_vis == 1, 0], X_vis[y_vis == 1, 1],c="#ef4444", alpha=0.35, s=25, label="Data Latih: Mengantuk")

        x0_min, x0_max = X_vis[:, 0].min() - 0.5, X_vis[:, 0].max() + 0.5
        x0_vals = np.linspace(x0_min, x0_max, 200)

        w, b = model_svm.w, model_svm.b
        ax.plot(x0_vals, (-w[0] * x0_vals + b) / w[1], "k-", lw=2, label="Hyperplane")
        ax.plot(x0_vals, (-w[0] * x0_vals + b + 1) / w[1], "k--", lw=1.1, label="Margin")
        ax.plot(x0_vals, (-w[0] * x0_vals + b - 1) / w[1], "k--", lw=1.1)

        if titik_scaled is not None:
            warna_titik = "#dc2626" if kelas_pred == 1 else "#16a34a"
            ax.scatter(titik_scaled[0], titik_scaled[1], c=warna_titik, edgecolors="black", marker="*", s=320, zorder=10, label=f"Input Pengemudi ({'+1' if kelas_pred == 1 else '-1'})")

        ax.set_xlim(x0_min, x0_max)
        ax.set_ylim(X_vis[:, 1].min() - 0.6, X_vis[:, 1].max() + 0.6)
        ax.set_title("Posisi Data Pengemudi terhadap Hyperplane", fontsize=10.5, fontweight="bold")
        ax.set_xlabel("Tingkat Theta (Scaled)", fontsize=9, fontweight="bold")
        ax.set_ylabel("Indeks Kedipan Mata (Scaled)", fontsize=9, fontweight="bold")
        ax.legend(loc="upper right", fontsize=7.5, frameon=True)
        ax.grid(True, linestyle=":", alpha=0.5)
        fig.tight_layout()
        canvas.draw()

    def proses_prediksi():
        try:
            theta_raw = float(entry_theta.get().strip())
            kedipan_raw = float(entry_kedipan.get().strip())
        except ValueError:
            messagebox.showerror("Input Tidak Valid", "Harap masukkan angka numerik yang valid untuk kedua sensor")
            return

        input_raw = np.array([theta_raw, kedipan_raw], dtype=float)
        input_scaled = (input_raw - MEAN_FITUR) / STD_FITUR

        nilai_fx = float(model_svm.decision(input_scaled))
        pred_kelas = int(model_svm.predict(input_scaled))
        jarak_hyperplane = abs(nilai_fx) / np.linalg.norm(model_svm.w)

        if pred_kelas == 1:
            hasil_box.configure(bg="#fef2f2")
            lbl_status.configure(text="Mengantuk", bg="#fef2f2", fg="#dc2626")
            lbl_rekomendasi.configure(text="Terdeteksi kelelahan.", bg="#fef2f2", fg="#991b1b"
            )
        else:
            hasil_box.configure(bg="#f0fdf4")
            lbl_status.configure(text="Tidak Mengantuk", bg="#f0fdf4", fg="#15803d")
            lbl_rekomendasi.configure(text="Pengemudi berada dalam kondisi terjaga.", bg="#f0fdf4", fg="#166534")

        lbl_detail.configure(text=(
            f"Detail Perhitungan Hyperplane:\n"
            f"Fitur Scaled            : [{input_scaled[0]:.3f}, {input_scaled[1]:.3f}]\n"
            f"Nilai f(x) = w·x - b    : {nilai_fx:+.4f}\n"
            f"Jarak ke Hyperplane     : {jarak_hyperplane:.4f} satuan"
            ))

        gambar_plot(titik_scaled=input_scaled, kelas_pred=pred_kelas)

    def reset_form():
        entry_theta.delete(0, tk.END)
        entry_kedipan.delete(0, tk.END)
        hasil_box.configure(bg="#f1f5f9")
        lbl_status.configure(text="Belum Dianalisis", bg="#f1f5f9", fg="#475569")
        lbl_rekomendasi.configure(text="Masukkan nilai sensor lalu klik Deteksi Status Kantuk", bg="#f1f5f9", fg="#64748b")
        lbl_detail.configure(text="Detail Perhitungan Hyperplane:\nFitur Scaled : -\nNilai f(x) = w·x - b  : -\nJarak ke Hyperplane   : -")
        gambar_plot()

    btn_frame = tk.Frame(left_panel, bg="#ffffff")
    btn_frame.pack(fill="x", side="bottom")

    tk.Button(
        btn_frame, text="Deteksi Status Kantuk", font=("Segoe UI", 10, "bold"),
        bg="#2563eb", fg="white", activebackground="#1d4ed8", activeforeground="white",
        relief="flat", pady=8, command=proses_prediksi
    ).pack(fill="x", pady=(0, 6))

    tk.Button(btn_frame, text="Reset Input", font=("Segoe UI", 9), bg="#e2e8f0", fg="#334155", relief="flat", pady=5, command=reset_form).pack(fill="x")

    proses_prediksi()
    root.mainloop()

jalankan_gui_svm()

---

## 8. Conclusion

Berdasarkan seluruh tahapan pengerjaan mulai dari *Business Understanding* sampai *GUI*, diperoleh kesimpulan akhir sebagai berikut:

1. **Pola Fisiologis Status Kantuk Pengemudi:**
   Kelelahan dan rasa kantuk pengemudi berkorelasi positif dengan peningkatan aktivitas gelombang otak *Tingkat Theta* serta berkorelasi negatif dengan *Indeks Kedipan Mata* per menit. Kedua fitur fisiologis ini memiliki pemisahan ruang fitur yang jelas sehingga sangat efektif diklasifikasikan menggunakan pendekatan batas lurus.

2. **Keberhasilan Pemodelan SVM:**
   Implementasi algoritma *Support Vector Machine* (SVM) menggunakan optimasi *Gradient Descent* pada *Hinge Loss* dan regularisasi berhasil menemukan *hyperplane* pemisah dengan margin yang lebar. Tanda koefisien bobot fitur pertama yang positif dan bobot fitur kedua yang negatif membuktikan bahwa model mempelajari arah hubungan secara tepat. Hasil pengujian pada data *testing* menunjukkan konsistensi generalisasi yang tinggi maupun kegagalan deteksi pada kelas positif.